In [ ]:
%pip install -e ../../../
%pip install plotly
%pip install nbformat

# Weather

https://www.knmi.nl/klimaat#klimaatdashboard



In [ ]:
from water_futures_battle.core.base_model import DynamicProperties
climate_dps = DynamicProperties('climate-dynamic_properties')

from water_futures_battle.climate.dynamic_properties import ClimateDB

import pandas as pd
from utility import upload_KNMI_csv_files_glob, rename_KNMI_scenarios, adjust_timestamp

DROP_COLUMNS_BASE = [
    'Hn-klimaatscenario',
    'Hn-klimaatscenario.2',
    'Hd-klimaatscenario',
    'Hd-klimaatscenario.2',
    'Ln-klimaatscenario',
    'Ln-klimaatscenario.2',
    'Ld-klimaatscenario',
    'Ld-klimaatscenario.2',
    ' ',
    'Trendlijn metingen',
    '90% band (low)',
    '90% band (high)',
    'Verwachting 2026'
]

## Temperature

We don't need the hourly or even daily values of temperature for our models.

We keep a few importnat markers of the time series.

### Average temperature in the season



In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Temperatuur',
    variables=[
        'Wintergemiddelde',
        'Lentegemiddelde',
        'Zomergemiddelde',
        'Herfstgemiddelde',
    ],
    drop_columns= DROP_COLUMNS_BASE
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2026 = pd.to_datetime('2025-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.TEMPERATURE_AVG] = df.copy() 

### Average minimum temperature

In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Gemiddelde-minimumtemperatuur',
    variables=[
        'Wintergemiddelde',
        'Lentegemiddelde',
        'Zomergemiddelde',
        'Herfstgemiddelde',
    ],
    drop_columns= DROP_COLUMNS_BASE
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2026 = pd.to_datetime('2025-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.TEMPERATURE_MIN_AVG] = df.copy() 

### Average maximum temperature

In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Gemiddelde-maximumtemperatuur',
    variables=[
        'Wintergemiddelde',
        'Lentegemiddelde',
        'Zomergemiddelde',
        'Herfstgemiddelde',
    ],
    drop_columns= DROP_COLUMNS_BASE
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2026 = pd.to_datetime('2025-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.TEMPERATURE_MAX_AVG] = df.copy() 

In [ ]:
# Extract highest temperature for each year between 2000 and 2024 (inclusive)
df_max_temp_per_year = df.loc[(df.index.year >= 2000) & (df.index.year <= 2024)]
df_max_temp_per_year = df_max_temp_per_year[['NL0000']].resample('YE').max()
df_max_temp_per_year['exponent'] = (df_max_temp_per_year['NL0000']/df_max_temp_per_year['NL0000'].min())**5
display(df_max_temp_per_year)
print("The mimnimum temperature in the historic period (reference temperature for the model is : ",df_max_temp_per_year['NL0000'].min())

import plotly.express as px

px.scatter((df_max_temp_per_year['NL0000']/df_max_temp_per_year['NL0000'].min())**5)

### Warmest day in a season


In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Warmste-dagen',
    variables=[
        'Warmste dag winter',
        'Warmste dag lente',
        'Warmste dag zomer',
        'Warmste dag herfst',
    ],
    drop_columns= DROP_COLUMNS_BASE
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2026 = pd.to_datetime('2025-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.TEMPERATURE_WARMDAY] = df.copy() 

### Coldest day in a season

In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Koudste-dagen',
    variables=[
        'Koudste dag winter',
        'Koudste dag lente',
        'Koudste dag zomer',
        'Koudste dag herfst',
    ],
    drop_columns= DROP_COLUMNS_BASE
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2026 = pd.to_datetime('2025-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.TEMPERATURE_COLDDAY] = df.copy() 

## Precipitation



In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Neerslagsom',
    variables=[
        'Hoeveelheid per winter',
        'Hoeveelheid per lente',
        'Hoeveelheid per zomer',
        'Hoeveelheid per herfst',
    ],
    drop_columns= DROP_COLUMNS_BASE
)

# Move the value in 'Verwachting deze Winter' to the actual observation
# WINTER2026 = pd.to_datetime('2025-12-01')
# FORECAST_COLUMN = 'Verwachting deze winter'
# df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
# df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

# Adjust to mm/day 
days_in_season = (df.index + pd.DateOffset(months=3) - df.index).days
df = df.div(days_in_season, axis=0)

df = adjust_timestamp(df)

climate_dps[ClimateDB.PRECIPITATION] = df.copy() 

## Solar radiation



In [ ]:
df = upload_KNMI_csv_files_glob(
    property='Zonnestraling',
    variables=[
        'Wintergemiddelde',
        'Lentegemiddelde',
        'Zomergemiddelde',
        'Herfstgemiddelde',
    ],
    drop_columns= DROP_COLUMNS_BASE+['Trend metingen']
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2026 = pd.to_datetime('2025-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2026, 'NL0000'] = df.loc[WINTER2026, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.SOLARRAD] = df.copy() 

## SPEI



In [ ]:
df = upload_KNMI_csv_files_glob(
    property='SPEI',
    variables=['Gemiddelde']*4,
    drop_columns= ['Trendlijn metingen', 'Verwachting 2025']
)

# Move the value in 'Verwachting deze Winter' to the actual observation
WINTER2025 = pd.to_datetime('2024-12-01')
FORECAST_COLUMN = 'Verwachting deze winter'
df.loc[WINTER2025, 'NL0000'] = df.loc[WINTER2025, FORECAST_COLUMN]
df.drop(columns=[FORECAST_COLUMN], inplace=True)

# df = rename_KNMI_scenarios(df)

df = adjust_timestamp(df)

climate_dps[ClimateDB.SPEI] = df.copy() 

In [ ]:
# Save out
climate_dps.dump(path='..')